# 

In [1]:
from ogb.utils.features import (allowable_features, atom_to_feature_vector,
 bond_to_feature_vector, atom_feature_vector_to_dict, bond_feature_vector_to_dict) 
import numpy as np
from tqdm import tqdm
import instructions_smol
import datasets
import pandas as pd
import os
from rdkit import Chem
from dataset_ood_download import get_data_list

/miniconda/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
train_path = '/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_train_gap_fixed_0224'
test_path = '/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_test_gap_fixed_0224'

In [3]:
# load data
train_data = datasets.load_from_disk(train_path)
test_data = datasets.load_from_disk(test_path)

In [4]:
train_data, test_data

(Dataset({
     features: ['task', 'x', 'edge_index', 'edge_attr', 'additional_x', 'additional_edge_index', 'additional_edge_attr', 'prompt_text', 'target_text', 'input_mol_string'],
     num_rows: 4961303
 }),
 Dataset({
     features: ['task', 'x', 'edge_index', 'edge_attr', 'additional_x', 'additional_edge_index', 'additional_edge_attr', 'prompt_text', 'target_text', 'input_mol_string'],
     num_rows: 32828
 }))

In [5]:
set(train_data['task'])

{'bace',
 'chebi-20-mol2text',
 'chebi-20-text2mol',
 'forward_reaction_prediction',
 'qm9_dipole_moment',
 'qm9_electronic_spatial_extent',
 'qm9_enthalpy_298K',
 'qm9_free_energy_298K',
 'qm9_heat_capacity_298K',
 'qm9_homo',
 'qm9_homo_lumo_gap',
 'qm9_internal_energy_298K',
 'qm9_isotropic_polarizability',
 'qm9_lumo',
 'qm9_zero_point_vibrational_energy',
 'reagent_prediction',
 'retrosynthesis',
 'smol-forward_synthesis',
 'smol-molecule_captioning',
 'smol-molecule_generation',
 'smol-name_conversion-i2f',
 'smol-name_conversion-i2s',
 'smol-name_conversion-s2f',
 'smol-name_conversion-s2i',
 'smol-property_prediction-bbbp',
 'smol-property_prediction-clintox',
 'smol-property_prediction-esol',
 'smol-property_prediction-hiv',
 'smol-property_prediction-lipo',
 'smol-property_prediction-sider',
 'smol-retrosynthesis'}

In [6]:
removing_tasks = [
 'qm9_dipole_moment',
 'qm9_electronic_spatial_extent',
 'qm9_enthalpy_298K',
 'qm9_free_energy_298K',
 'qm9_heat_capacity_298K',
 'qm9_internal_energy_298K',
 'qm9_isotropic_polarizability',
 'qm9_zero_point_vibrational_energy',
 'smol-name_conversion-i2f',
 'smol-name_conversion-s2f',
]

In [11]:
filtered_test_data = test_data.filter(lambda x: x['task'] not in removing_tasks
                 , num_proc=16)
filtered_test_data

Dataset({
    features: ['task', 'x', 'edge_index', 'edge_attr', 'additional_x', 'additional_edge_index', 'additional_edge_attr', 'prompt_text', 'target_text', 'input_mol_string'],
    num_rows: 32828
})

In [12]:
filtered_train_data = train_data.filter(lambda x: x['task'] not in removing_tasks,
                  num_proc=200)
filtered_train_data

Dataset({
    features: ['task', 'x', 'edge_index', 'edge_attr', 'additional_x', 'additional_edge_index', 'additional_edge_attr', 'prompt_text', 'target_text', 'input_mol_string'],
    num_rows: 3306674
})

In [13]:
filtered_train_data.save_to_disk(
    '/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_train_3.3M_0415'
)
filtered_test_data.save_to_disk(
    '/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_test_3.3M_0415'
)

Saving the dataset (1/1 shards): 100%|██████████| 32828/32828 [00:03<00:00, 10212.58 examples/s]
